# Answer height estimate: calibrating from the markdown

The check that keeps an answer from being cut off runs in the orchestrator, where it has only the model's `answer_markdown`. So this notebook estimates height **from the markdown**, with a handful of plain constants, and tunes those constants against heights read from the app.

**The model**
- text: a cost per character, with heading characters weighted for the larger font
- tables: a cost per row (the header counts as a row)
- code blocks: a fixed cost per line
- charts: a cost per image, fitted (standard charts measure about 613, a beeswarm about 1,000; the markdown can't tell them apart)

**What it does**
1. Reads each conversation's turns twice: the chat history (to pair by question and check the HTML length) and the findings ledger (the markdown).
2. Computes features from the markdown AST, the same walk the orchestrator will use.
3. Fits the constants to the measured heights, printing measured against estimated and a leave-one-out error.
4. Estimates every turn of both conversations and shows how the answers over the limit would be split into parts.

Needs your credentials (BigQuery reads). Run top to bottom.

In [ ]:
import sys
sys.path.insert(0, "..")

import math

import mistune
import numpy as np

from app.gateway.chat_history.reader import get_conversation_history
from app.orchestrator.conversation_memory.reader import fetch_turns

LIMIT = 4000                  # the height one answer may have
CODE_LINE = 30                # fixed per code line: no measured answer has a code block
HEADING_CHAR_WEIGHT = 1.33    # headings are 32px against 24px body text

# heights read from the app (turn numbers are positions in the chat history) and the HTML lengths shown beside them
CONVERSATIONS = {
    "e27f378a-5158-49e4-9e2c-79b3f971789a": {
        "heights": {1: 836, 2: 1180, 3: 2007, 4: 1502, 5: 2712, 6: 804, 7: 708, 8: 708, 9: 2886, 10: 676, 11: 2885,
                    12: 1586, 13: 1713, 14: 740, 15: 1592, 26: 6674},
        "html": {1: 3403, 2: 3913, 3: 4026, 4: 3547, 5: 5519, 6: 2009, 7: 1797, 8: 1753, 9: 5533, 10: 1767, 11: 5470,
                 12: 12760, 13: 13011, 14: 2099, 15: 3395, 26: 17723},
        "clamped": [27, 28],      # read 7,630 (the control's cap): cut off, so the real height is at least that
    },
    "9a28b70a-b757-4ad1-b375-a8a9a49a3c9a": {
        "heights": {3: 5756},
        "html": {3: 9343},
        "clamped": [],
    },
}

In [ ]:
rows = []          # one per chat-history turn that has a ledger row: conversation, turn number, question, markdown, height
unmatched = []
for conversation_id, info in CONVERSATIONS.items():
    chat_turns = await get_conversation_history(conversation_id)
    ledger = await fetch_turns(conversation_id, 1, 100000, False)
    available = list(ledger)
    for number, turn in enumerate(chat_turns, start=1):
        match = next((r for r in available if r["question"] == turn.question), None)
        if match is None:
            unmatched.append((conversation_id[:8], number, turn.question[:50]))
            continue
        available.remove(match)
        rows.append({"conversation": conversation_id, "number": number, "question": turn.question,
                     "markdown": match["answer_markdown"], "html_len": len(turn.answer_html),
                     "height": info["heights"].get(number)})
    print(f"{conversation_id[:8]}: {len(chat_turns)} chat turns, {len(ledger)} ledger rows")

print("\nturns with no ledger row (declined or unverified), left out:", unmatched or "none")

problems = [(r["conversation"][:8], r["number"], r["html_len"], CONVERSATIONS[r["conversation"]]["html"][r["number"]])
            for r in rows if r["number"] in CONVERSATIONS[r["conversation"]]["html"]
            and r["html_len"] != CONVERSATIONS[r["conversation"]]["html"][r["number"]]]
print("html length mismatches against what you read (conversation, turn, stored, read):", problems or "none")

In [ ]:
markdown_ast = mistune.create_markdown(renderer="ast", plugins=["table"])


def inline_parts(nodes) -> tuple[str, int]:
    """The visible text of inline nodes, and how many images they hold."""
    text, images = "", 0
    for node in nodes:
        kind = node["type"]
        if kind in ("text", "codespan", "inline_html"):
            text += node.get("raw", "")
        elif kind in ("softbreak", "linebreak"):
            text += " "
        elif kind == "image":
            images += 1
        else:
            inner_text, inner_images = inline_parts(node.get("children", []))
            text, images = text + inner_text, images + inner_images
    return text, images


def node_features(node) -> dict:
    """What sets a block's height. A list is one block whose items are summed."""
    f = {"text_chars": 0, "heading_chars": 0, "text_blocks": 0, "headings": 0, "rows": 0, "code_lines": 0, "images": 0}
    kind = node["type"]
    if kind in ("paragraph", "block_text"):
        text, images = inline_parts(node["children"])
        f.update(text_chars=len(text), text_blocks=1, images=images)
    elif kind == "heading":
        text, images = inline_parts(node["children"])
        f.update(heading_chars=len(text), headings=1, images=images)
    elif kind == "block_code":
        f["code_lines"] = node["raw"].count("\n") + 1
    elif kind == "table":
        body = next((c for c in node["children"] if c["type"] == "table_body"), None)
        f["rows"] = 1 + (len(body["children"]) if body else 0)
    elif kind in ("list", "list_item", "block_quote"):
        for child in node["children"]:
            for key, value in node_features(child).items():
                f[key] += value
    return f


def label_of(node) -> str:
    kind = node["type"]
    if kind in ("heading", "paragraph", "block_text"):
        return inline_parts(node["children"])[0].strip()[:60] or "(chart)"
    return {"list": "a list", "table": "a table", "block_code": "a code block"}.get(kind, kind)


def blocks_of(markdown: str) -> list[dict]:
    """The answer's top-level blocks, each with its features, in order."""
    return [{"label": label_of(node), "is_heading": node["type"] == "heading", "features": node_features(node)}
            for node in markdown_ast(markdown) if node["type"] != "blank_line"]


def totals(blocks: list[dict]) -> dict:
    return {key: sum(b["features"][key] for b in blocks) for key in blocks[0]["features"]} if blocks else \
        {"text_chars": 0, "heading_chars": 0, "text_blocks": 0, "headings": 0, "rows": 0, "code_lines": 0, "images": 0}


NAMES = ["per character", "per table row", "per chart"]


def vector(blocks: list[dict]) -> list[float]:
    t = totals(blocks)
    return [t["text_chars"] + HEADING_CHAR_WEIGHT * t["heading_chars"], t["rows"], t["images"]]


def fixed_part(blocks: list[dict]) -> float:
    t = totals(blocks)
    return CODE_LINE * t["code_lines"]

In [ ]:
def fit_nonnegative(X, y):
    """Least squares through the origin, dropping any constant that comes out negative and refitting."""
    active = list(range(X.shape[1]))
    while True:
        coefficients, *_ = np.linalg.lstsq(X[:, active], y, rcond=None)
        negative = [k for k, c in zip(active, coefficients) if c < 0]
        if not negative:
            break
        worst = min(negative, key=lambda k: coefficients[active.index(k)])
        print(f"dropping '{NAMES[worst]}' (fitted negative)")
        active.remove(worst)
    full = np.zeros(X.shape[1])
    full[active] = coefficients
    return full


measured = [r for r in rows if r["height"] is not None]
for r in rows:
    r["blocks"] = blocks_of(r["markdown"])
X = np.array([vector(r["blocks"]) for r in measured])
y = np.array([r["height"] - fixed_part(r["blocks"]) for r in measured], dtype=float)
COEFFICIENTS = fit_nonnegative(X, y)

rmse = float(np.sqrt(np.mean((X @ COEFFICIENTS - y) ** 2)))
print(f"fitted on {len(measured)} measured answers, rmse {rmse:.0f} units")
print(f"fixed: code line {CODE_LINE}, heading characters x {HEADING_CHAR_WEIGHT}\n")
for name, value in zip(NAMES, COEFFICIENTS):
    print(f"  {name:<16}{value:>9.3f}")

In [ ]:
def estimate(blocks: list[dict]) -> float:
    return float(np.array(vector(blocks)) @ COEFFICIENTS + fixed_part(blocks))


print(f"{'conv':>8} {'turn':>4} {'measured':>9} {'estimated':>10} {'error':>7}  charts rows  question")
errors = []
for r in measured:
    estimated = estimate(r["blocks"])
    t = totals(r["blocks"])
    errors.append(estimated - r["height"])
    print(f"{r['conversation'][:8]:>8} {r['number']:>4} {r['height']:>9.0f} {estimated:>10.0f} {estimated - r['height']:>+7.0f}  {t['images']:>6} {t['rows']:>4}  {r['question'][:40]}")

loo = []
for left_out in range(len(measured)):
    keep = [i for i in range(len(measured)) if i != left_out]
    coefficients = fit_nonnegative(X[keep], y[keep])
    loo.append(float(X[left_out] @ coefficients - y[left_out]))
print(f"\nfit error: mean absolute {np.mean(np.abs(errors)):.0f}, worst under {-min(errors):.0f}, worst over {max(errors):.0f}")
print(f"leave-one-out: mean absolute {np.mean(np.abs(loo)):.0f}, worst under {-min(loo):.0f}, worst over {max(loo):.0f}")

print("\nanswers that were cut off at the control's cap (real height at least 7,630):")
for r in rows:
    if r["number"] in CONVERSATIONS[r["conversation"]]["clamped"]:
        print(f"  {r['conversation'][:8]} turn {r['number']}: estimated {estimate(r['blocks']):.0f}")

In [ ]:
def plan_parts(blocks: list[dict], limit: float) -> list[dict]:
    """Packs blocks into parts that each fit the limit, cutting before a heading when one is available."""
    base = 0.0
    heights = [estimate_block(b) for b in blocks]
    parts, current, height = [], [], base
    for index, block_height in enumerate(heights):
        while current and height + block_height > limit:
            headings = [k for k, i in enumerate(current) if blocks[i]["is_heading"] and k > 0]
            cut = headings[-1] if headings else len(current)
            parts.append(current[:cut])
            current = current[cut:]
            height = base + sum(heights[i] for i in current)
        current.append(index)
        height += block_height
    if current:
        parts.append(current)
    return [{"first": blocks[p[0]]["label"], "height": base + sum(heights[i] for i in p),
             "too_big": len(p) == 1 and base + heights[p[0]] > limit} for p in parts]


def estimate_block(block: dict) -> float:
    f = block["features"]
    chars = f["text_chars"] + HEADING_CHAR_WEIGHT * f["heading_chars"]
    return COEFFICIENTS[0] * chars + COEFFICIENTS[1] * f["rows"] + COEFFICIENTS[2] * f["images"] + CODE_LINE * f["code_lines"]


print(f"limit {LIMIT}. Every answer over it, with the parts it would be sent in:\n")
for r in rows:
    estimated = estimate(r["blocks"])
    if estimated <= LIMIT:
        continue
    parts = plan_parts(r["blocks"], LIMIT)
    print(f"{r['conversation'][:8]} turn {r['number']}: about {estimated:.0f} units -> {len(parts)} parts   ({r['question'][:50]})")
    for number, part in enumerate(parts, start=1):
        big = "   (this block alone is over the limit)" if part["too_big"] else ""
        print(f"    part {number}: starts at '{part['first'][:50]}', about {part['height']:.0f}{big}")

over = sum(1 for r in rows if estimate(r["blocks"]) > LIMIT)
print(f"\n{over} of {len(rows)} answers are over the limit")

In [ ]:
# The orchestrator's estimator, copied from the graph notebook, checked against the same heights.
# Uses `rows` and `CONVERSATIONS` from the cells above.
_markdown_ast = markdown_ast

# Answer height estimate, fitted to heights read from the app (app units).
MAX_ANSWER_HEIGHT = 4000
HEIGHT_PER_CHAR = 0.6
HEIGHT_PER_TABLE_ROW = 42
HEIGHT_PER_CHART = 700
HEIGHT_PER_CODE_LINE = 30
HEADING_CHAR_WEIGHT = 1.33     # headings are 32px against 24px body text


def inline_text_and_images(nodes: list[dict]) -> tuple[str, int]:
    """Visible text and image count of inline nodes."""
    text, images = "", 0
    for node in nodes:
        if node["type"] == "image":
            images += 1
        elif "children" in node:
            inner_text, inner_images = inline_text_and_images(node["children"])
            text, images = text + inner_text, images + inner_images
        else:
            text += node.get("raw", "")
    return text, images


def block_height(node: dict) -> float:
    kind = node["type"]
    if kind in ("paragraph", "block_text", "heading"):
        text, images = inline_text_and_images(node["children"])
        weight = HEADING_CHAR_WEIGHT if kind == "heading" else 1
        return HEIGHT_PER_CHAR * weight * len(text) + HEIGHT_PER_CHART * images
    if kind == "block_code":
        return HEIGHT_PER_CODE_LINE * (node["raw"].rstrip("\n").count("\n") + 1)
    if kind == "table":
        body = next((c for c in node["children"] if c["type"] == "table_body"), None)
        return HEIGHT_PER_TABLE_ROW * (1 + (len(body["children"]) if body else 0))
    return sum(block_height(child) for child in node.get("children", []))


def answer_height(answer_markdown: str) -> float:
    return sum(block_height(node) for node in _markdown_ast(answer_markdown))


def block_label(node: dict) -> str:
    """What a block starts with, for naming where an answer is cut."""
    if node["type"] in ("heading", "paragraph", "block_text"):
        return inline_text_and_images(node["children"])[0][:60] or "a chart"
    return {"list": "a list", "table": "a table", "block_code": "a code block"}.get(node["type"], node["type"])


def split_answer_into_sections(answer_markdown: str) -> list[dict]:
    """Splits the answer into consecutive sections that each fit MAX_ANSWER_HEIGHT, cutting before a
    heading when one is available. Each section is {"starts_with": its first block, "height": its estimated height}."""
    block_list = []
    for markdown_node in _markdown_ast(answer_markdown):
        if markdown_node["type"] == "blank_line":
            continue
        block_list.append({
            "label": block_label(markdown_node),
            "is_heading": markdown_node["type"] == "heading",
            "height": block_height(markdown_node),
        })

    answer_sections = []
    height_left = sum(block["height"] for block in block_list)
    while len(block_list) > 1 and height_left > MAX_ANSWER_HEIGHT:
        # index of the first block where the running height passes the limit
        running_height = 0
        for first_over_limit_index, block in enumerate(block_list):
            running_height += block["height"]
            if running_height > MAX_ANSWER_HEIGHT:
                break
        first_over_limit_index = max(first_over_limit_index, 1)    # a block taller than the limit gets a section to itself

        # index where the next section begins: the last heading before that block, else the block itself
        cut_index = first_over_limit_index
        for index in range(1, first_over_limit_index):
            if block_list[index]["is_heading"]:
                cut_index = index

        section_blocks = block_list[:cut_index]
        section_height = sum(block["height"] for block in section_blocks)
        answer_sections.append({"starts_with": section_blocks[0]["label"], "height": section_height})
        block_list = block_list[cut_index:]
        height_left -= section_height
    answer_sections.append({"starts_with": block_list[0]["label"], "height": height_left})

    return answer_sections


In [ ]:
print(f"{'conv':>8} {'turn':>4} {'measured':>9} {'estimated':>10} {'error':>7}  question")
errors = []
for r in measured:
    estimated = answer_height(r["markdown"])
    errors.append(estimated - r["height"])
    print(f"{r['conversation'][:8]:>8} {r['number']:>4} {r['height']:>9.0f} {estimated:>10.0f} {estimated - r['height']:>+7.0f}  {r['question'][:40]}")
print(f"\nmean absolute error {np.mean(np.abs(errors)):.0f}, worst under {-min(errors):.0f}, worst over {max(errors):.0f}")

print("\nanswers cut off at the control's cap (real height at least 7,630):")
for r in rows:
    if r["number"] in CONVERSATIONS[r["conversation"]]["clamped"]:
        print(f"  {r['conversation'][:8]} turn {r['number']}: estimated {answer_height(r['markdown']):.0f}")

print(f"\nevery answer estimated over {MAX_ANSWER_HEIGHT}, with its sections:")
for r in rows:
    height = answer_height(r["markdown"])
    if height <= MAX_ANSWER_HEIGHT:
        continue
    answer_sections = split_answer_into_sections(r["markdown"])
    print(f"{r['conversation'][:8]} turn {r['number']} {r['question'][:40]!r}: about {height:,.0f} -> {len(answer_sections)} sections")
    for number, section in enumerate(answer_sections, start=1):
        over = "   <-- over the limit" if section["height"] > MAX_ANSWER_HEIGHT else ""
        print(f"    section {number}: '{section['starts_with'][:50]}', about {section['height']:,.0f}{over}")


## Reading the results

- **The pairing lines** in the third cell should show no HTML length mismatches. A turn with no ledger row is a declined or unverified turn and is left out.
- **The fit** is good when the errors are small next to the heights, the constants are positive, and the leave-one-out error is close to the fit error. With so few measured answers, trust the leave-one-out number.
- **The cut-off answers** must estimate at or above 7,630. If they don't, the estimate is low for tall answers and the limit needs more margin.
- **The split plan** is the real output: every answer over the limit, with where each part would start. A 10,000-unit answer should come out as three parts.
- The chart constant is fitted. Expect about 613 (a standard chart); a beeswarm is about 1,000, so an answer with one is estimated low. Pick the orchestrator's value with that in mind, and keep the margin in the 4,000 limit.
- If the fonts or the pane width change, remeasure and run this again.